# 08 Lab RFDC + F-engine FFT Debug

Interactive bring-up notebook for the current lab setup:

- no external PPS
- no external 10 MHz
- ADC0 and DAC0 connected
- no CMAC/network path required
- observe F-engine debug time waveform and hardware FFT power bins from PYNQ

Run the cells in order. Use the buttons in the control cell instead of `Run All` when debugging hardware state.

Troubleshooting notes from board bring-up:

- After `Init lab RFDC`, expect `rfdc_status_flags = 0x1f`, `streaming = 1`, and a growing `rfdc_sample_count`.
- `TimeoutError: DEBUG_STATUS=0x00000000` means the debug observer never saw a completed RFDC capture; reload the current bitstream, run `Init lab RFDC`, and confirm `rfdc_adc_valid = 1` before capture.
- Old overlays may report `DEBUG_STATUS = 0x00000006`: bit2 means capture done and the buffers are valid, while bit1 was an over-strict observer error flag. New Stage 1 RTL targets a clean successful status of `0x00000004` after the overlay is rebuilt.

## 1. Import and Locate Overlay

This cell expects the notebook to live under `/home/xilinx/jupyter_notebooks/t510_fengine` on the board. It also works from the repository checkout if the `overlay/` and `python/` directories are present.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/home/xilinx/jupyter_notebooks/t510_fengine'),
    Path('/home/xilinx/t510_fengine_bringup'),
    Path('/home/astrolab/demo-ant'),
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'overlay' / 't510_fengine.bit').exists() and (root / 'python' / 't510_fengine.py').exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find overlay/t510_fengine.bit and python/t510_fengine.py')

sys.path.insert(0, str(PROJECT_ROOT))
from python.t510_fengine import T510FEngine

BITFILE = PROJECT_ROOT / 'overlay' / 't510_fengine.bit'
print('Python:', sys.executable)
print('Project root:', PROJECT_ROOT)
print('Bitfile:', BITFILE)
print('Expected core version: 0x00010002')

## 2. Interactive Control Panel

Recommended first pass:

1. Keep `ADC active mask = ADC0 only`.
2. Keep `sync = free_run` via `Init lab RFDC`.
3. Click `Load overlay`.
4. Click `Init lab RFDC`.
5. Click `Capture time` and `Capture spectrum`.
6. Change `DAC tone phase step`, click `Apply DAC tone`, then capture spectrum again.

In [ ]:
state = {'core': None, 'last_time': None, 'last_spectrum': None}

mask_widget = W.Dropdown(
    options=[
        ('ADC0 only: 0x0001', 0x0001),
        ('ADC0 I/Q candidate: 0x0003', 0x0003),
        ('All RFDC ports: 0xffff', 0xffff),
    ],
    value=0x0001,
    description='ADC mask',
    style={'description_width': '120px'},
)
mode_widget = W.Dropdown(
    options=['snapshot', 'time', 'spec', 'dual'],
    value='snapshot',
    description='F-engine mode',
    style={'description_width': '120px'},
)
tone_enable_widget = W.Checkbox(value=True, description='DAC tone enable')
tone_amplitude_widget = W.IntSlider(
    value=2048,
    min=0,
    max=8192,
    step=128,
    description='Tone amplitude',
    style={'description_width': '120px'},
)
tone_phase_step_widget = W.Text(
    value='0x00800000',
    description='Phase step',
    style={'description_width': '120px'},
)
timeout_widget = W.FloatSlider(
    value=2.0,
    min=0.2,
    max=10.0,
    step=0.2,
    description='Timeout s',
    style={'description_width': '120px'},
)
download_widget = W.Checkbox(value=True, description='Download bitstream on load')

load_button = W.Button(description='Load overlay', button_style='primary')
init_button = W.Button(description='Init lab RFDC', button_style='success')
status_button = W.Button(description='Read status')
tone_button = W.Button(description='Apply DAC tone')
time_button = W.Button(description='Capture time')
spectrum_button = W.Button(description='Capture spectrum')
stop_button = W.Button(description='Stop')
reset_button = W.Button(description='Soft reset')
out = W.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})


def parse_int_text(text):
    return int(str(text).replace('_', ''), 0)


def core():
    if state['core'] is None:
        raise RuntimeError('Click Load overlay first')
    return state['core']


def print_status(status):
    hex_keys = {
        'core_version', 'sync_config', 'status', 'rfdc_status_flags',
        'rfdc_active_mask', 'rfdc_current_valid_mask', 'rfdc_seen_valid_mask',
        'debug_status', 'dac_tone_phase_step'
    }
    keys = [
        'core_version', 'sync_config', 'configured_sync_mode', 'configured_clock_ref',
        'status', 'armed', 'streaming', 'waiting_for_epoch', 'fsm_state',
        'rfdc_status_flags', 'rfdc_core_ready', 'rfdc_adc_valid', 'rfdc_dac_ready', 'rfdc_clock_locked',
        'rfdc_active_mask', 'rfdc_current_valid_mask', 'rfdc_seen_valid_mask', 'rfdc_sample_count',
        'monitor_sample_count', 'debug_status', 'debug_done', 'debug_error', 'debug_capture_count',
        'debug_nfft', 'debug_sample_rate_hz', 'debug_peak_bin', 'debug_peak_power',
        'dac_tone_control', 'dac_tone_amplitude', 'dac_tone_phase_step',
    ]
    for key in keys:
        if key not in status:
            continue
        value = int(status[key])
        if key in hex_keys or 'mask' in key:
            print(f'{key:28s}: 0x{value:08x}')
        else:
            print(f'{key:28s}: {value}')
    if int(status.get('core_version', 0)) != 0x00010002:
        print('\nWARNING: core_version is not 0x00010002. You may be running an older bitstream.')


def on_load(_):
    with out:
        clear_output(wait=True)
        print('Loading overlay...')
        state['core'] = T510FEngine(str(BITFILE), download=bool(download_widget.value))
        print('Overlay loaded.')
        print('Overlay IP keys:', list(getattr(state['core'].overlay, 'ip_dict', {}).keys()))
        print_status(state['core'].read_status())


def on_init(_):
    with out:
        clear_output(wait=True)
        print('Configuring lab mode: clock_ref=tcxo_10mhz, sync_mode=free_run')
        status = core().init_lab_rfdc(
            mask=int(mask_widget.value),
            mode=str(mode_widget.value),
            tone_enable=bool(tone_enable_widget.value),
            tone_amplitude=int(tone_amplitude_widget.value),
            tone_phase_step=parse_int_text(tone_phase_step_widget.value),
            wait_seconds=float(timeout_widget.value),
        )
        print_status(status)


def on_status(_):
    with out:
        clear_output(wait=True)
        print_status(core().read_status())


def on_tone(_):
    with out:
        clear_output(wait=True)
        step = parse_int_text(tone_phase_step_widget.value)
        core().set_dac_tone(
            enable=bool(tone_enable_widget.value),
            amplitude=int(tone_amplitude_widget.value),
            phase_step=step,
        )
        print(f'DAC tone applied: enable={tone_enable_widget.value}, amplitude={tone_amplitude_widget.value}, phase_step=0x{step:08x}')
        print_status(core().read_status())


def on_time(_):
    with out:
        clear_output(wait=True)
        samples = np.asarray(core().capture_time(timeout=float(timeout_widget.value)))
        state['last_time'] = samples
        print(f'Captured time samples: shape={samples.shape}, dtype={samples.dtype}')
        print(f'I min/max/std: {samples[:,0].min()} / {samples[:,0].max()} / {samples[:,0].std():.2f}')
        print(f'Q min/max/std: {samples[:,1].min()} / {samples[:,1].max()} / {samples[:,1].std():.2f}')
        print('First 16 IQ samples:')
        for idx, (ival, qval) in enumerate(samples[:16]):
            print(f'  {idx:04d}: I={int(ival):6d} Q={int(qval):6d}')
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(samples[:, 0], label='I')
        ax.plot(samples[:, 1], label='Q')
        ax.set_title('ADC0 debug time capture')
        ax.set_xlabel('sample index at observer rate')
        ax.set_ylabel('ADC code')
        ax.grid(True)
        ax.legend()
        plt.show()


def on_spectrum(_):
    with out:
        clear_output(wait=True)
        spec = core().capture_spectrum(timeout=float(timeout_widget.value))
        state['last_spectrum'] = spec
        power = np.asarray(spec['power'])
        freq_mhz = np.asarray(spec['freq_hz']) / 1e6
        peak_bin = int(spec['peak_bin'])
        peak_power = int(spec['peak_power'])
        peak_mhz = float(freq_mhz[peak_bin]) if 0 <= peak_bin < len(freq_mhz) else float('nan')
        print(f'FFT bins: {len(power)}')
        print(f'Observer sample rate: {int(spec["sample_rate_hz"])} Hz')
        print(f'Peak bin: {peak_bin}')
        print(f'Peak frequency, unshifted: {peak_mhz:.6f} MHz')
        print(f'Peak power: {peak_power}')
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(freq_mhz, power)
        ax.axvline(peak_mhz, color='tab:red', alpha=0.7, label=f'peak bin {peak_bin}')
        ax.set_title('Hardware FFT debug spectrum, unshifted')
        ax.set_xlabel('frequency (MHz)')
        ax.set_ylabel('power')
        ax.grid(True)
        ax.legend()
        plt.show()


def on_stop(_):
    with out:
        clear_output(wait=True)
        core().stop()
        time.sleep(0.05)
        print_status(core().read_status())


def on_reset(_):
    with out:
        clear_output(wait=True)
        core().reset()
        time.sleep(0.05)
        print_status(core().read_status())


load_button.on_click(on_load)
init_button.on_click(on_init)
status_button.on_click(on_status)
tone_button.on_click(on_tone)
time_button.on_click(on_time)
spectrum_button.on_click(on_spectrum)
stop_button.on_click(on_stop)
reset_button.on_click(on_reset)

controls = W.VBox([
    W.HTML('<b>T510 F-engine lab RFDC debug controls</b>'),
    W.HBox([download_widget, tone_enable_widget]),
    W.HBox([mask_widget, mode_widget, timeout_widget]),
    W.HBox([tone_amplitude_widget, tone_phase_step_widget]),
    W.HBox([load_button, init_button, status_button, tone_button]),
    W.HBox([time_button, spectrum_button, stop_button, reset_button]),
    out,
])
display(controls)

## 3. Non-widget Fallback

If buttons do not render in your browser, run this cell manually. It performs the same basic sequence with ADC0 only.

In [ ]:
# Optional fallback: uncomment and run if widgets are unavailable.
# core = T510FEngine(str(BITFILE), download=True)
# status = core.init_lab_rfdc(mask=0x1, mode='snapshot', tone_enable=True, tone_amplitude=2048, tone_phase_step=0x00800000, wait_seconds=2.0)
# print_status(status)
# samples = np.asarray(core.capture_time(timeout=2.0))
# spec = core.capture_spectrum(timeout=2.0)
# print(samples[:16])
# print('peak_bin', spec['peak_bin'], 'peak_power', spec['peak_power'])

## Expected Results

- `core_version` should be `0x00010002`.
- `configured_sync_mode` should become `2` after `Init lab RFDC`, meaning `free_run`.
- `configured_clock_ref` should become `1`, meaning `tcxo_10mhz`.
- `streaming` should become `1` without PPS.
- `rfdc_current_valid_mask & ADC mask` should be nonzero if RFDC ADC data is valid.
- `rfdc_sample_count` should increase between status reads.
- Time capture should not be all zeros or all constant.
- Spectrum peak should be stable, and should move when `Phase step` is changed and `Apply DAC tone` is clicked.

If `core_version` is wrong, reload the latest `overlay/t510_fengine.bit`. If `pynq` import fails, open the notebook from the board's normal Jupyter server so it uses `/usr/local/share/pynq-venv/bin/python`.